<a href="https://colab.research.google.com/github/ljcheng84/JC-NUS-ISS-ARCHAAS-STE-2026/blob/main/JC_Workshop_1_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SETUP AND INSTALL LIBRARIES

In [4]:
!pip install -q openai

# TESTING FOR OPENAI_API_KEY VALIDITY

In [24]:
import os
import json
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Literal

def load_openai_key():
    """Read the OpenAI API key from Colab Secrets; ask for it only if that fails."""
    try:
        from google.colab import userdata
        return userdata.get("OPENAI_API_KEY")
    except ImportError:
        print("Not running in Colab.")
    except Exception as e:
        # e.g. the secret is missing, or Notebook access is turned off
        print(f"Could not read the Colab secret OPENAI_API_KEY ({type(e).__name__}).")
        print("Check the steps above: secret name, value, and Notebook access.")
    from getpass import getpass
    return getpass("Enter your OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = load_openai_key()
client = OpenAI()

# Use a non-reasoning model for this demo:
# reasoning models do not accept the temperature setting used in section 7.
MODEL = "gpt-4o-mini"
print("OpenAI client is ready. Model:", MODEL)

OpenAI client is ready. Model: gpt-4o-mini


# CREATING CUSTOM CLASS AND FUNCTIONS

In [27]:
def process_order(customer_input):
    response = client.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that extracts food orders into structured JSON format."},
            {"role": "user", "content": f"Parse this order: {customer_input}"}
        ],
        temperature=0.1,  # Low temperature for consistent output
        response_format=Order, # This enforces strict JSON matching your schema
    )
    return json.loads(response.choices[0].message.content)

#Define the exact JSON schema using Pydantic
class OrderItem(BaseModel):
    name: str
    quantity: int
    size: Literal["regular", "small", "large"]

class Order(BaseModel):
    items: List[OrderItem]
    total_items: int

TEST INPUTS WITH SAMPLE ORDERS

In [29]:
# Test with various inputs
orders = [
    "2x Ice Latte, 1 large fries and 3 hamburger",
    "I'll have four cappuccino and seven croissants please",
    "Please give me 5 Ice Milo, 7 Happy Meal and 20-piece nuggets",
    "Can I have 3 Kopi-O, 2 large bak-chor-mee",
    "Give me 4 chawamushi, 2 cold soba, 3 large set of sashimi"
    # Add more test cases
]

for order in orders:
    print(f"Input: {order}")
    print(f"Output: {process_order(order)}")
    print()

Input: 2x Ice Latte, 1 large fries and 3 hamburger
Output: {'items': [{'name': 'Ice Latte', 'quantity': 2, 'size': 'regular'}, {'name': 'fries', 'quantity': 1, 'size': 'large'}, {'name': 'hamburger', 'quantity': 3, 'size': 'regular'}], 'total_items': 6}

Input: I'll have four cappuccino and seven croissants please
Output: {'items': [{'name': 'cappuccino', 'quantity': 4, 'size': 'regular'}, {'name': 'croissant', 'quantity': 7, 'size': 'regular'}], 'total_items': 11}

Input: Please give me 5 Ice Milo, 7 Happy Meal and 20-piece nuggets
Output: {'items': [{'name': 'Ice Milo', 'quantity': 5, 'size': 'regular'}, {'name': 'Happy Meal', 'quantity': 7, 'size': 'regular'}, {'name': '20-piece nuggets', 'quantity': 1, 'size': 'regular'}], 'total_items': 13}

Input: Can I have 3 Kopi-O, 2 large bak-chor-mee
Output: {'items': [{'name': 'Kopi-O', 'quantity': 3, 'size': 'regular'}, {'name': 'bak-chor-mee', 'quantity': 2, 'size': 'large'}], 'total_items': 5}

Input: Give me 4 chawamushi, 2 cold soba, 3